# LlamaIndex와 AgentCore 메모리 - 의료 지식 어시스턴트 (단기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 의료 지식 어시스턴트를 만드는 방법을 보여줍니다. 단일 환자 상담 세션 내에서의 **단기 메모리** 지속성에 초점을 맞추며, 의료 상담 전반에 걸쳐 환자 증상, 병력, 약물 상호작용 및 진단 추론을 기억할 수 있도록 합니다.

## 아키텍처 개요

![LlamaIndex AgentCore 단기 메모리 아키텍처](LlamaIndex-AgentCore-STM-Arch.png)

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형 메모리                                                               |
| 에이전트 사용 사례  | 의료 지식 어시스턴트                                                             |
| 에이전트 프레임워크 | LlamaIndex                                                                       |
| LLM 모델            | Anthropic Claude 3.7 Sonnet                                                       |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, LlamaIndex 에이전트, 의료 분석 도구                       |
| 예제 난이도         | 중급                                                                             |

학습 내용:
- 의료 상담 데이터를 위한 AgentCore 메모리 생성
- 의료 워크플로를 위한 LlamaIndex 네이티브 메모리 통합 사용
- 환자 분석을 위한 의료 전용 도구 구축
- 단일 상담 세션 내에서 의료 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 컨텍스트

이 예제에서는 의료 제공자가 단일 상담 세션 내에서 환자 사례를 분석하고, 약물 상호작용을 확인하며, 임상 가이드라인을 조회하는 "의료 지식 어시스턴트"를 만듭니다. 이 어시스턴트는 AgentCore 메모리를 사용하여 상담 전반에 걸쳐 환자 증상, 병력, 약물 및 진단 추론에 대한 컨텍스트를 유지합니다.

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 의존성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore 메모리 구성

의료 어시스턴트를 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# AgentCore 메모리 리소스 생성
region = os.getenv('AWS_REGION', 'us-east-1')
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f'MedicalAssistantShortTerm_{int(datetime.now().timestamp())}',
        description='Medical knowledge assistant short-term memory for single consultation context',
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )
    memory_id = response['id']
    print(f"AgentCore 메모리 생성 완료: {memory_id}")
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 의료 분석 도구 구현

의료 상담 작업을 위한 전문 도구를 정의합니다:

In [ ]:
def record_patient_symptoms(symptoms: str, severity: str, duration: str) -> str:
    """Record patient symptoms with severity and duration"""
    print(f"증상 기록: {symptoms} (중증도: {severity}, 기간: {duration})")
    return f"Recorded patient symptoms: {symptoms}"

def check_drug_interaction(medication1: str, medication2: str, interaction_level: str) -> str:
    """Check drug interaction between medications"""
    print(f"약물 상호작용 확인: {medication1} + {medication2} (위험도: {interaction_level})")
    return f"Drug interaction assessed: {medication1} and {medication2}"

def save_vital_signs(temperature: str, blood_pressure: str, heart_rate: str, notes: str) -> str:
    """Save patient vital signs with notes"""
    print(f"활력 징후: 체온 {temperature}, 혈압 {blood_pressure}, 심박수 {heart_rate}")
    return f"Saved vital signs for patient"

def retrieve_clinical_guideline(condition: str, guideline_type: str, evidence_level: str) -> str:
    """Retrieve clinical guideline for medical condition"""
    print(f"{condition}에 대한 {guideline_type} 가이드라인 조회 (근거 수준: {evidence_level})")
    return f"Retrieved clinical guideline for {condition}"

def document_differential_diagnosis(primary_diagnosis: str, alternatives: str, confidence: str) -> str:
    """Document differential diagnosis with confidence level"""
    print(f"감별 진단: {primary_diagnosis} (신뢰도: {confidence})")
    return f"Documented differential diagnosis: {primary_diagnosis}"

# 에이전트용 도구 객체 생성
medical_tools = [
    FunctionTool.from_defaults(fn=record_patient_symptoms),
    FunctionTool.from_defaults(fn=check_drug_interaction),
    FunctionTool.from_defaults(fn=save_vital_signs),
    FunctionTool.from_defaults(fn=retrieve_clinical_guideline),
    FunctionTool.from_defaults(fn=document_differential_diagnosis)
]

## 4단계: LlamaIndex 에이전트 구현

단기 메모리 컨텍스트를 가진 의료 어시스턴트 에이전트를 생성합니다:

In [ ]:
# 단기 메모리 구성 (단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="medical-provider",
    memory_id=memory_id,
    session_id="consultation-session-today",  # 전체에서 동일한 세션
    namespace="/medical-consultation/"
)

# AgentCore 메모리 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 의료 어시스턴트 에이전트 생성
medical_agent = FunctionAgent(
    tools=medical_tools,
    llm=llm,
    verbose=True
)

print("단기 메모리가 적용된 의료 지식 어시스턴트 준비 완료!")

## 5단계: 단기 메모리 기능 테스트

포괄적인 환자 상담 세션을 통해 의료 어시스턴트의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 환자 접수 및 초기 평가

In [ ]:
# 환자 세부 정보로 상담 세션 초기화
# (Emily Chen 의사가 환자 John Smith(45세 남성)를 상담 - 흉통/호흡곤란/피로 증상, 중증, 3일간 지속, 고혈압 및 당뇨 병력)
response = await medical_agent.run(
    "I'm Dr. Emily Chen conducting a consultation for patient John Smith, 45-year-old male. "
    "Record symptoms: 'chest pain, shortness of breath, fatigue' with 'severe' severity and '3 days' duration. "
    "Patient has history of hypertension and diabetes.",
    memory=agentcore_memory
)

print("환자 접수:")
print(response)

### 테스트 2: 활력 징후 기록

In [ ]:
# 임상 컨텍스트와 함께 활력 징후 기록
# (체온 99.2°F, 혈압 165/95 mmHg, 심박수 110 bpm, 혈압 상승 및 빈맥, 환자 발한 및 불안 상태)
response = await medical_agent.run(
    "Save vital signs: temperature '99.2°F', blood pressure '165/95 mmHg', heart rate '110 bpm' "
    "with notes 'elevated BP and tachycardia, patient appears diaphoretic and anxious'.",
    memory=agentcore_memory
)

print("활력 징후 기록:")
print(response)

### 테스트 3: 약물 상호작용 분석

In [ ]:
# 현재 복용 약물의 약물 상호작용 확인
# (리시노프릴 10mg과 메트포르민 500mg - 낮은 상호작용 수준, 고혈압 및 당뇨 관리에 모두 복용 중)
response = await medical_agent.run(
    "Check drug interaction between 'Lisinopril 10mg' and 'Metformin 500mg' with 'low' interaction level. "
    "Patient is currently taking both for hypertension and diabetes management.",
    memory=agentcore_memory
)

print("약물 상호작용 확인:")
print(response)

# 잠재적 신규 약물 상호작용 확인
# (리시노프릴 10mg과 니트로글리세린 설하정 - 중간 상호작용 수준, 흉통 관리를 위한 니트로글리세린 투여 고려)
response = await medical_agent.run(
    "Check drug interaction between 'Lisinopril 10mg' and 'Nitroglycerin sublingual' with 'moderate' interaction level. "
    "Considering nitroglycerin for chest pain management.",
    memory=agentcore_memory
)

print("추가 약물 확인:")
print(response)

### 테스트 4: 환자 컨텍스트 회상

In [ ]:
# 환자 정보 및 활력 징후 회상 테스트
# (어떤 환자를 상담 중인지, 증상, 활력 징후, 현재 약물 질문)
response = await medical_agent.run(
    "What patient am I consulting with? What are their presenting symptoms, vital signs, and current medications?",
    memory=agentcore_memory
)

print("환자 컨텍스트 회상:")
print(response)
print("\n예상 결과: John Smith, 45세 남성, 흉통/호흡곤란/피로, 혈압 165/95, 리시노프릴/메트포르민")

### 테스트 5: 임상 가이드라인 조회

In [ ]:
# 증상에 기반한 임상 가이드라인 조회
# (급성 흉통에 대한 진단 프로토콜 - 근거 수준 A, 환자의 흉통을 체계적으로 평가 필요)
response = await medical_agent.run(
    "Retrieve clinical guideline for 'acute chest pain' with 'diagnostic protocol' type and 'Level A' evidence level. "
    "Need to evaluate this patient's chest pain systematically.",
    memory=agentcore_memory
)

print("임상 가이드라인 조회:")
print(response)

### 테스트 6: 감별 진단 기록

In [ ]:
# 추론이 포함된 감별 진단 기록
# (주 진단: 급성 관상동맥 증후군, 대안: 폐색전증/대동맥 박리/불안 장애, 높은 신뢰도 - 흉통/활력 징후 상승/심장 위험 인자 기반)
response = await medical_agent.run(
    "Document differential diagnosis: primary 'Acute Coronary Syndrome' with alternatives "
    "'pulmonary embolism, aortic dissection, anxiety disorder' and 'high' confidence level. "
    "Based on chest pain, elevated vitals, and cardiac risk factors.",
    memory=agentcore_memory
)

print("감별 진단:")
print(response)

### 테스트 7: 종합 임상 추론

In [ ]:
# 종합 임상 추론 테스트
# (John의 증상, 활력 징후, 병력을 기반으로 급성 관상동맥 증후군을 고려한 이유와 이를 뒷받침하는 구체적 임상 지표 질문)
response = await medical_agent.run(
    "Based on John's symptoms, vital signs, and medical history, why did I consider Acute Coronary Syndrome? "
    "What specific clinical indicators support this diagnosis?",
    memory=agentcore_memory
)

print("임상 추론 테스트:")
print(response)
print("\n예상 결과: 흉통 + 호흡곤란 + 혈압/심박수 상승 + 당뇨/고혈압 병력 = ACS 위험 인자")

### 테스트 8: 약물 상호작용 회상

In [ ]:
# 약물 상호작용 메모리 테스트
# (이 환자에 대해 확인한 약물 상호작용과 중간 위험이었던 조합 및 이유 질문)
response = await medical_agent.run(
    "What drug interactions have I checked for this patient? Which combination had moderate risk and why?",
    memory=agentcore_memory
)

print("약물 상호작용 회상:")
print(response)
print("\n예상 결과: 리시노프릴+메트포르민 (낮은 위험), 리시노프릴+니트로글리세린 (중간 위험)")

### 테스트 9: 치료 계획 통합

In [ ]:
# 통합 치료 계획 테스트
# (감별 진단과 약물 상호작용 확인을 기반으로, John에 대해 염두에 두어야 할 치료 고려사항 - 약물 상호작용 및 임상 가이드라인 포함)
response = await medical_agent.run(
    "Based on my differential diagnosis and drug interaction checks, what treatment considerations "
    "should I keep in mind for John? Include medication interactions and clinical guidelines.",
    memory=agentcore_memory
)

print("치료 계획:")
print(response)
print("\n예상 결과: ACS 프로토콜, 리시노프릴+니트로글리세린 상호작용 모니터링, 심장 정밀 검사 고려")

In [ ]:
# 종합 사례 요약
# (환자 인구통계, 증상, 활력 징후, 현재 약물, 확인된 약물 상호작용, 감별 진단, 조회된 임상 가이드라인의 완전한 사례 요약 요청)
response = await medical_agent.run(
    "Provide a complete case summary: patient demographics, presenting symptoms, vital signs, "
    "current medications, drug interactions checked, differential diagnosis, and clinical guidelines retrieved.",
    memory=agentcore_memory
)

print("종합 사례 요약:")
print(response)
print("\n예상 결과: 기록된 모든 정보가 포함된 전체 상담 세부 사항")

## 6단계: 세션 경계 테스트

다른 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다:

In [ ]:
# 다른 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="medical-provider",
    memory_id=memory_id,
    session_id="different-consultation-session",  # 다른 세션 ID
    namespace="/medical-consultation/"
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
# (오늘 어떤 환자를 상담 중인지, 어떤 증상과 활력 징후를 기록했는지 질문)
response = await medical_agent.run(
    "What patients am I consulting with today? What symptoms and vital signs have I recorded?",
    memory=new_session_memory
)

print("세션 경계 테스트 (다른 세션):")
print(response)
print("\n예상 결과: 이전 세션에서의 회상이 제한적이거나 없음 (단기 메모리 경계)")

In [ ]:
# 원래 세션으로 돌아가서 지속성 확인
# (원래 상담으로 돌아와서 John Smith의 정확한 활력 징후와 주 진단 질문)
response = await medical_agent.run(
    "Back in my original consultation - what were John Smith's exact vital signs and primary diagnosis?",
    memory=agentcore_memory  # 원래 세션 메모리
)

print("원래 세션 복귀:")
print(response)
print("\n예상 결과: 혈압 165/95, 심박수 110, ACS 진단 전체 회상")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀들을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 세션 초반 정보를 회상할 수 있는지 확인"""
        has_content = len(response) > 50
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내에서 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동합니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하며, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 중요한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상
response1 = await medical_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리
response2 = await medical_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 기능
response3 = await medical_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

### 테스트 10: 종합 사례 요약

## 요약

이 노트북에서 다음을 시연했습니다:

- **단기 메모리 통합**: 세션 범위 의료 상담을 위해 LlamaIndex와 AgentCore 메모리 사용

- **의료 전용 도구**: 환자 증상 추적, 약물 상호작용 확인 및 임상 가이드라인 조회

- **임상 추론**: 어시스턴트가 환자 세부 사항, 활력 징후 및 진단 추론을 기억

- **약물 안전 관리**: 종합적인 약물 상호작용 추적 및 평가

- **세션 경계**: 서로 다른 환자 상담 세션 간의 메모리 격리

- **근거 기반 의학**: 임상 가이드라인 통합 및 감별 진단 기록

의료 지식 어시스턴트는 단기 메모리가 어떻게 단일 상담 세션 내에서 포괄적인 환자 케어를 가능하게 하면서 서로 다른 환자 진료 간에 명확한 경계를 유지하는지 보여줍니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다:

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")